## Installing necessary libraries

Cohere provides free trial keys to use their LLMs. So generate one trial key from dashboard.cohere.com

In [ ]:
!pip uninstall -y pydantic pydantic-core
!pip install -U pydantic langchain langchain-core langchain-community langchain-cohere langchain-classic chromadb pdfminer.six


langchain-cohere: Enables integration of Cohere's language models with LangChain for advanced text generation and processing workflows.

langchain: Provides a modular framework for building language model-powered applications, such as chatbots, question-answering systems, and conversational agents.

pdfminer.six: Facilitates text extraction from PDF files, making it useful for document analysis and preprocessing tasks.

chromadb: A vector database library designed for efficient storage and retrieval of embeddings, ideal for tasks like semantic search and recommendation systems.

## Importing libraries

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
COHERE_API_KEY=os.getenv("COHERE_API_KEY")


In [4]:

import os
from typing import List

from pydantic import BaseModel, Field

from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.documents import Document

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.vectorstores import Chroma

from langchain_cohere import ChatCohere, CohereEmbeddings

from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_text_splitters import RecursiveCharacterTextSplitter

from pdfminer.high_level import extract_text as extract_text_pdf_miner


## VectorDB setup

In [22]:
# Define the directory where the Chroma database will persist data

persist_directory = "chroma_db"

# Initialize Cohere embeddings with the specified model
# "embed-english-v3.0" is a pre-trained English language embedding model by Cohere
# The user_agent parameter specifies the tool or library using the Cohere API, in this case, LangChain
embedding = CohereEmbeddings(
    model="embed-english-v3.0",
    user_agent="langchain"
)

We are processing 2 research papers on transformers and yolo. You can use the PDFs

In [19]:
download_dir = "papers"
os.makedirs(download_dir, exist_ok=True)

papers = {
    "1706.03762v7.pdf": "https://arxiv.org/pdf/1706.03762v7.pdf",
    "1506.02640v5.pdf": "https://arxiv.org/pdf/1506.02640v5.pdf",
}

for filename, url in papers.items():
    output_path = os.path.join(download_dir, filename)
    urllib.request.urlretrieve(url, output_path)
    print(f"Downloaded: {output_path}")

Downloaded: papers/1706.03762v7.pdf
Downloaded: papers/1506.02640v5.pdf


In [23]:
# Loop through a list of PDF files to process
for pdf_name in [
    "papers/1706.03762v7.pdf",
    "papers/1506.02640v5.pdf"
]:
    # Open each PDF file in binary mode
    with open(pdf_name, 'rb') as f:
        # Extract text from the PDF using the extract_text_pdf_miner function
        text = extract_text_pdf_miner(f)

        # Clean the extracted text by removing newline characters and joining into a single string
        cleaned_text = " ".join(text.split("\n"))

        # Initialize a list to store document chunks
        docs = []

        # Create a text splitter to divide the text into manageable chunks
        # Each chunk has a maximum size of 2048 characters with a 512-character overlap
        splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

        # Split the cleaned text into chunks and wrap each chunk in a Document object
        for chunk in splitter.split_text(cleaned_text):
            docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))

    # Create a Chroma collection from the processed documents
    # Use the specified persist directory and embedding model for storage and retrieval
    vector_collection_fixed_size = Chroma.from_documents(
        documents=docs,
        persist_directory=persist_directory,
        embedding=embedding
    )

In [24]:
# Initialize a Chroma vector database
# The persist_directory specifies the location where the database is stored
# The embedding_function parameter provides the embedding model used for vector representation
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

/var/folders/1g/yz_fh_zd53g6vjqvt5f81krc0000gn/T/ipykernel_15443/1816146847.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)


In [25]:
# Perform a similarity search on the vector database
# The query "What is YOLO?" is used to find the most relevant documents
# k=1 specifies that the top 1 most similar document should be retrieved
# The method also returns relevance scores indicating how closely each document matches the query
vectordb.similarity_search_with_relevance_scores("What is YOLO?", k=1)

[(Document(metadata={'source': 'papers/1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In real-world applications it is hard to predict all possible use cases and  YOLO is a fast, accurate object detector, making it ideal for computer vision applications. We connect YOLO to a webcam and verify that it maintains real-time performance,  \x0cVOC 2007 AP 59.2 54.2 43.2 36.5 -  Picasso AP Best F1 0.590 53.3 0.226 10.4 0.458 37.8 0.271 17.8 0.051 1.9  People-Art AP 45 26 32  YOLO R-CNN DPM Poselets [2] D&T [4]  (a) Picasso Dataset precision-recall curves.  (b) Quantitative results on the VOC 2007, Picasso, and People-Art Datasets. The Picasso Dataset evaluates on both AP and best F1 score.  Figure 5: Generalization results on Picasso and People-Art datasets.  Figure 6: Qualitative Results. YOLO running on sample artwork and natural images from the internet. It is mostly accurate alt

## Chain Setup

In [30]:
# Initialize an LLM instance using Cohere's "command-r" model
# The temperature parameter controls randomness in the generated responses; 0 ensures deterministic outputs
llm = ChatCohere(model="command-a-03-2025")

In [32]:
# Define a prompt template for generating answers based on a given context and question
prompt_str = """Given a chat history and the latest user question which might reference context in the chat history,
formulate a standalone question which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.
"""

# Use a prompt that includes a MessagesPlaceholder variable under the name "chat_history".
# This allows us to pass in a list of Messages to the prompt using the "chat_history" input key,
# and these messages will be inserted after the system message and before the human message containing the
# latest question.

prompt_history_aware = ChatPromptTemplate.from_messages([
    ("system", prompt_str),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# create_history_aware_retriever constructs a chain that accepts keys input and chat_history as input, and has the same output schema as a retriever
history_aware_retriever = create_history_aware_retriever(
    llm, vectordb.as_retriever(), prompt_history_aware
)

Now its time to build our final rag_chain with create_retrieval_chain. This chain applies the history_aware_retriever and question_answer_chain created with create_stuff_documents_chain in sequence, retaining intermediate outputs such as the retrieved context for convenience. It has input keys input and chat_history, and includes input, chat_history, context, and answer in its output.



In [33]:
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question. If you don't know the answer,
say that you don't know. Use three sentences maximum and keep theanswer concise.

{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# create_retrieval_chain combines history aware retriever and the qa chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [34]:
# Load a empty chat_history list
chat_history = []

# Invoking the chain
question = "What is YOLO?"
response = rag_chain.invoke({"input": question, "chat_history": chat_history})

# Appeding question and response answers
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response["answer"]),
    ]
)

In [35]:
#check chat_history
chat_history

[HumanMessage(content='What is YOLO?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='YOLO (You Only Look Once) is a unified, real-time object detection model that predicts bounding boxes and class probabilities directly from image pixels. It is fast, simple, and generalizes well to new domains, making it ideal for applications requiring robust, real-time detection.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [36]:
second_question = "What are the tasks YOLO can be used in?"
response = rag_chain.invoke({"input": second_question, "chat_history": chat_history})

print(response["answer"])

YOLO (You Only Look Once) is a versatile object detection system that can be used in various tasks, particularly those requiring fast and accurate real-time object detection. Key applications include:

1. **Autonomous Driving**: Detecting vehicles, pedestrians, and traffic signs in real-time to enable safe navigation.  
2. **Assistive Devices**: Providing real-time scene information to visually impaired users.  
3. **Robotics**: Enabling robots to perceive and interact with their environment efficiently.  
4. **Webcam and Video Analysis**: Tracking objects in live video streams, such as monitoring activities or detecting anomalies.  
5. **Artwork and Domain Generalization**: Detecting objects in non-natural images like artwork, where it outperforms other methods.  

YOLO’s speed and accuracy make it ideal for these and other computer vision applications.


To automate the inserting and updating of chat history. And have a session id that can be unique for a user

In [37]:
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [38]:
conversational_rag_chain.invoke(
    {"input": "What is Transformers?"},
    config={
        "configurable": {"session_id": "User_1"}
    },
)["answer"]

'Transformers are a model architecture in sequence modeling and transduction tasks, such as machine translation, that rely entirely on self-attention mechanisms to draw global dependencies between input and output, eschewing recurrence and allowing for significantly more parallelization during training. They achieve state-of-the-art performance with reduced training time and computational costs compared to recurrent models.'

In [39]:
# This will output input, chat_history, contexts and answer
conversational_rag_chain.invoke(
    {"input": "Transformers vs YOLO"},
    config={
        "configurable": {"session_id": "User_1"}
    },
)

{'input': 'Transformers vs YOLO',
 'chat_history': [HumanMessage(content='What is Transformers?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Transformers are a model architecture in sequence modeling and transduction tasks, such as machine translation, that rely entirely on self-attention mechanisms to draw global dependencies between input and output, eschewing recurrence and allowing for significantly more parallelization during training. They achieve state-of-the-art performance with reduced training time and computational costs compared to recurrent models.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(metadata={'source': 'papers/1506.02640v5.pdf'}, page_content='Search [35] generates potential bounding boxes, a convolu- tional network extracts features, an SVM scores the boxes, a linear model adjusts the bounding boxes, and non-max sup- pression eliminates duplicate detections. Each stage of this 

In [40]:
# Check the chat_history of the store dict
for message in store["User_1"].messages:
    if isinstance(message, AIMessage):
        prefix = "AI"
    else:
        prefix = "User"

    print(f"{prefix}: {message.content}\n")

User: What is Transformers?

AI: Transformers are a model architecture in sequence modeling and transduction tasks, such as machine translation, that rely entirely on self-attention mechanisms to draw global dependencies between input and output, eschewing recurrence and allowing for significantly more parallelization during training. They achieve state-of-the-art performance with reduced training time and computational costs compared to recurrent models.

User: Transformers vs YOLO

AI: Transformers and YOLO (You Only Look Once) are both influential architectures in computer vision, but they serve different purposes and have distinct characteristics:

1. **Purpose**:  
   - **YOLO** is primarily designed for **object detection**, focusing on identifying and localizing objects within an image in real-time.  
   - **Transformers**, originally developed for natural language processing, have been adapted for **vision tasks** (e.g., Vision Transformers or ViTs) and excel in tasks like imag